# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam271/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [17]:
# ML-04 — Warehouse setup
%pip -q install duckdb huggingface_hub

import os
import getpass
import duckdb
import pandas as pd
import numpy as np

# Get HF token from Colab Secret / environment.
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# For ML-04, use the March 2026 partition as the mid-panel month.
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

print("DuckDB connected.")
print("Verification month: March 2026")

Paste your Hugging Face READ token (hf_...): ··········
DuckDB connected.
Verification month: March 2026


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: One row represents one content page for one client on one report date.
Time window: For this contract, I use March 2026 as the mid-panel verification window. The broader warehouse contains daily content-performance history across the available reporting period.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1 — Verify the grain
grain_check = con.execute(f"""
SELECT
    COUNT(*) AS rows_total,
    COUNT(DISTINCT
        report_date || '|' || client_hash_id || '|' || content_hash_id
    ) AS unique_grain_keys
FROM {MARCH}
""").df()

display(grain_check)

assert grain_check.loc[0, "rows_total"] == grain_check.loc[0, "unique_grain_keys"], \
    "Grain check failed: duplicate report_date × client × content rows found."

print("Grain verified: one row = one report_date × client × content.")

,rows_total,unique_grain_keys
0,9841378,9841378


Grain verified: one row = one report_date × client × content.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature: gsc_impressions, gsc_clicks, gsc_avg_position, ga4_pageviews, ga4_sessions — historical performance signals that can be used as predictors when they are measured before the decision moment.

Label: The downstream content-performance outcome/proxy derived from future observations. It is not used as an input feature.

Context: report_date, client_hash_id, content_hash_id, client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available — identifiers, date, and data-availability context.

Excluded: Future/outcome-derived fields and any fields from after the decision moment, because they would leak information that would not be available when making the prediction.

In [19]:
# Section 2 — Verify the fields used in the contract

selected_fields = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "client_has_gsc",
    "client_has_ga4",
    "gsc_data_available",
    "ga4_data_available",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
]

field_check = columns_check[
    columns_check["column_name"].isin(selected_fields)
].copy()

display(field_check)

assert set(selected_fields).issubset(set(columns_check["column_name"])), \
    "One or more selected fields are missing."

assert len(field_check) == len(selected_fields), \
    "Expected all 13 selected fields to be present."

print(f"Verified {len(selected_fields)} contract fields.")

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
10,gsc_avg_position,DOUBLE,YES,None,None,None


Verified 12 contract fields.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Grain:** The March data contains 9,841,378 rows and 9,841,378 unique `report_date × client_hash_id × content_hash_id` keys, confirming one row represents one report date × client × content observation.

**Row count and date window:** The March partition contains 9,841,378 rows, covering `2026-03-01` through `2026-03-31`.

**Availability:** Of the 9,841,378 March rows, 3,611,061 have GSC data available and 413,966 have GA4 data available, using `IS TRUE` to explicitly check availability.

These checks verify the grain, time window, row count, and data availability of the March slice using the real warehouse data.


In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3 — Verify grain, counts/date span, and availability

# Query 1 — Grain
grain_check = con.execute(f"""
SELECT
    COUNT(*) AS rows_total,
    COUNT(DISTINCT
        report_date || '|' || client_hash_id || '|' || content_hash_id
    ) AS unique_grain_keys
FROM {MARCH}
""").df()

display(grain_check)

assert grain_check.loc[0, "rows_total"] == grain_check.loc[0, "unique_grain_keys"], \
    "Grain check failed."

print("1) Grain verified.")


# Query 2 — Row count and date span
count_window = con.execute(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {MARCH}
""").df()

display(count_window)

print("2) Row count and date span verified.")


# Query 3 — Availability using IS TRUE
availability_check = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM {MARCH}
""").df()

display(availability_check)

print("3) Availability verified using IS TRUE.")

,rows_total,unique_grain_keys
0,9841378,9841378


1) Grain verified.


,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


2) Row count and date span verified.


,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


3) Availability verified using IS TRUE.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### 4. Data limits

This slice has several limitations: history may be unbalanced across clients and content, some early observations may have GSC data only, and overlapping time windows can make observations dependent on nearby periods. Availability also varies across rows, so missing GSC or GA4 data can limit which signals are usable for some observations. These data limitations mean the results should be treated as **observed and directional**, not causal.


In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 — Verify data limits

limits_check = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available,
    COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS gsc_unavailable,
    COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS ga4_unavailable
FROM {MARCH}
""").df()

display(limits_check)

print("Data-limit checks completed for the March slice.")

,total_rows,gsc_available,ga4_available,gsc_unavailable,ga4_unavailable
0,9841378,3611061,413966,6230317,9427412


Data-limit checks completed for the March slice.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.